# 📘 Session 3: LangChain Fundamentals, Local Models & Agents

This notebook explores the core abstractions of **LangChain**:
- Setting up observability with **LangSmith**
- Interacting with local LLMs using **ChatOllama** (`llama3.2:3b`)
- Standardized message types (`SystemMessage`, `HumanMessage`, `AIMessage`)
- Multi-turn conversation handling
- Defining reusable tools with `@tool` and schema validation via **Pydantic** (`args_schema`)
- Autonomous agent execution using `create_agent`

## 1. Observability Setup (LangSmith)
Enable tracing to monitor model invocations, latency, and token consumption.

In [3]:
import os
from dotenv import load_dotenv

# Load credentials and enable LangSmith tracing
load_dotenv()
os.environ["LANGSMITH_TRACING"] = "true"
if not os.environ.get("LANGSMITH_API_KEY"):
    import getpass
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

print("LangSmith tracing configured.")

## 2. LangChain Message Abstractions & Local Model
LangChain normalizes prompts into structured message objects:
- `SystemMessage`: Governs instructions, persona, and behavioral boundaries.
- `HumanMessage`: Represents user input.
- `AIMessage`: Represents the model's response.

In [4]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage
from langchain.tools import tool
from langchain_ollama import ChatOllama


## 3. Basic Model Invocation (Grammar Corrector)
Test basic prompt completion using local Ollama (`llama3.2:3b`).

In [6]:
# System and user messages
system_msg = (
    "You are an English grammar corrector. Review the provided text and correct "
    "grammatical, spelling, and punctuation errors. Make it clear and straightforward "
    "without changing the core meaning."
)
human_msg = "youre son is not studying properly , he is sleeping while i am giving lecture"

messages = [SystemMessage(system_msg), HumanMessage(human_msg)]

# Initialize local Ollama model (temperature=0.7 for balanced creativity and precision)
model = ChatOllama(
    model="llama3.2:3b",
    temperature=0.7,
    base_url="http://127.0.0.1:11434"
)

resp = model.invoke(messages)
print(resp.text)

The corrected text would be:

"Your son is not studying properly; he is sleeping while I am giving the lecture."


In [7]:
resp.content

'The corrected text would be:\n\n"Your son is not studying properly; he is sleeping while I am giving the lecture."'

## 4. Multi-Turn Conversation History
LLMs are stateless. To support multi-turn dialogue, prior messages (`AIMessage` and `HumanMessage`) must be appended to the conversation context list.

In [8]:
# Demonstrating conversation continuation
messages = [
    SystemMessage(system_msg),
    HumanMessage(human_msg),
    AIMessage(resp.content),
    HumanMessage("i want to say for daughter not for son")
]
response = model.invoke(messages)
print(response.text)

You're right, it's typically more common to use "daughter" instead of "son" in such cases. Here's the corrected text:

"I'm worried that my daughter isn't studying properly; she's sleeping while I'm giving the lecture."

(I've made no changes to the part of speech, only replacing "son" with "daughter" as per your request)


In [9]:
# Demonstrating manual message history for stateful recall
history = [HumanMessage("my name is abhi")]
res = model.invoke(history)
print("Response 1:", res.text)

history.append(AIMessage(res.text))
history.append(HumanMessage("what is my name"))

res = model.invoke(history)
print("Response 2:", res.text)

Abhi! It's nice to meet you! Is there something I can help you with, or would you like to chat?
[HumanMessage(content='my name is abhi', additional_kwargs={}, response_metadata={}), AIMessage(content="Abhi! It's nice to meet you! Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Your name is Abhi! I remembered from our initial conversation. Is everything okay?


In [10]:
history.append(HumanMessage("do you support tool calling?"))
res = model.invoke(history)
print(res.text)

I know your name is Abhi!

And yes, I can support tool calling, also known as "API" (Application Programming Interface) interactions. This means I can:

* Provide documentation for specific tools or platforms
* Offer instructions on how to use certain tools or APIs
* Help troubleshoot issues related to tool usage

Please specify the tool you're interested in, and I'll do my best to assist you, Abhi!


## 5. Tool Definition & LangChain Agents (`create_agent`)
Tools allow language models to interface with external systems (APIs, databases, services). The `@tool` decorator automatically inspects function type annotations and docstrings to create an OpenAI/LangChain compatible tool definition.

In [15]:
from langchain.agents import create_agent

@tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery database to find the delivery status of an order."""
    return f"Order {order_id} Ready to ship"

@tool
def get_refund_policy() -> str:
    """Return the company return and refund policy."""
    return "You can return any product within 7 days of purchase."

ollama_model = ChatOllama(model="llama3.2:3b")

# Construct agent equipped with domain tools
agent = create_agent(
    model=ollama_model,
    tools=[get_order_status, get_refund_policy],
    system_prompt="You are the Sourav Services customer support assistant. Use tools when needed."
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "who are you? Where is my order 22?"}
]})
result

{'messages': [HumanMessage(content='who are you? Where is my order 22?', additional_kwargs={}, response_metadata={}, id='e80a7d0a-098e-458a-8fc2-f83c06e364c5'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-10-30T06:13:46.425192Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4771672500, 'load_duration': 3654570083, 'prompt_eval_count': 213, 'prompt_eval_duration': 677066000, 'eval_count': 15, 'eval_duration': 415736000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--01a18b71-5087-739d-b402-81f5e819f4f1-0', tool_calls=[{'name': 'get_order_status', 'args': {'order_id': '22'}, 'id': '097d094f-3d2b-45b0-b358-b3c17ef9598d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 213, 'output_tokens': 15, 'total_tokens': 228}),
  ToolMessage(content='Order22 Ready to ship', name='get_order_status', id='37f7ee16-3adf-4fd2-a2d5-40f4398d8956', tool_call_id='09

In [16]:
print(result["messages"][-1].text)

Hello! I'm Sourav from Sourav Services. I'm here to help you with your order.

Regarding your order #22, I have checked on the status, and it is currently ready to ship. You can expect to receive your order within the estimated delivery time frame.

Here are the details of your order:

* Order Number: #22
* Status: Ready to Ship
* Estimated Delivery Time: [insert estimated delivery time]

If you have any questions or concerns about your order, please don't hesitate to contact us. We're here to help!

Is there anything else I can assist you with?


## 6. Schema-Enforced Tools (`args_schema`)
For complex tools with multiple arguments, default values, and strict validation, pass a Pydantic `BaseModel` to `@tool(args_schema=...)`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input schema for weather queries."""
    location: str = Field(description="City name or geographic coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Whether to include a 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather conditions and optional multi-day forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units.capitalize()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [24]:
# Initialize agent with the structured weather tool
weather_agent = create_agent(
    model=ollama_model,
    tools=[get_weather]
)

result = weather_agent.invoke({"messages": [
    {"role": "user", "content": "whats the weather now in dhaka?, include forecast, what day is today?"}
]})
print(result["messages"][-1].text)

Today is Tuesday, March 21, 2023.

Here's a brief forecast for the next 5 days:

* Today: Partly Cloudy with a high of 25°C and a low of 18°C
* Tomorrow: Sunny with a high of 28°C and a low of 20°C
* Wednesday: Cloudy with a high of 24°C and a low of 19°C
* Thursday: Light Rain with a high of 23°C and a low of 18°C
* Friday: Sunny with a high of 26°C and a low of 20°C

Please note that the forecast is subject to change and may not be accurate. For the most up-to-date and accurate weather information, I recommend checking a reliable weather source.


## 7. Key Takeaways
- **Standardization:** LangChain's message formats simplify switching between local (Ollama) and cloud (Gemini, OpenAI) models.
- **Tool Binding:** `@tool` turns standard Python functions into LLM-callable actions.
- **Agent Reasoning:** `create_agent` orchestrates tool selection, parameter extraction, and output synthesis automatically.